In [1]:

from datasets import load_dataset
import random
import pandas as pd

# set random seed for reproducibility
random.seed(42)

ds = load_dataset("locuslab/TOFU", "full")["train"]
ds

Dataset({
    features: ['question', 'answer'],
    num_rows: 4000
})

In [2]:

# TOFU: 200 fictitious authors, 20 consecutive QA pairs each
N_PER_AUTHOR = 20
n_authors = len(ds) // N_PER_AUTHOR

# Group into author blocks: author_groups[i][k] = k-th question for author i
author_groups = []
for i in range(n_authors):
    start = i * N_PER_AUTHOR
    author_groups.append([ds[j] for j in range(start, start + N_PER_AUTHOR)])

# Extract author name from questions
def extract_author_name(rows):
    names = []
    for row in rows:
        q = row["question"]
        if "'s " not in q:
            continue
        idx = q.rfind("'s ")
        chunk = q[:idx]
        words = chunk.split()
        name_parts = []
        for w in reversed(words):
            if w[0].isupper() or w.lower() in ("de", "van", "von", "del", "la", "el", "al") or "-" in w:
                name_parts.insert(0, w)
            else:
                break
        if len(name_parts) >= 2:
            names.append(" ".join(name_parts))
    if not names:
        return "Unknown"
    from collections import Counter
    return Counter(names).most_common(1)[0][0]

author_names = [extract_author_name(g) for g in author_groups]

# Pick 5 random authors, all 20 questions each
chosen_idx = random.sample(range(n_authors), 5)

mcqs = []
for ai in chosen_idx:
    for k in range(N_PER_AUTHOR):
        row = author_groups[ai][k]
        correct = row["answer"]

        other_authors = [x for x in range(n_authors) if x != ai]
        distractor_authors = random.sample(other_authors, 3)
        distractors = [author_groups[da][k]["answer"] for da in distractor_authors]

        options = distractors + [correct]
        random.shuffle(options)
        answer_idx = chr(65 + options.index(correct))

        mcqs.append({
            "author": author_names[ai],
            "question": row["question"],
            "A": options[0],
            "B": options[1],
            "C": options[2],
            "D": options[3],
            "answer": answer_idx,
        })

df = pd.DataFrame(mcqs)
df.to_csv("store/good.csv", index=False)

In [3]:
print(f"Authors: {[author_names[i] for i in chosen_idx]}\n")

for _, row in df.iterrows():
    print(f"Author: {row['author']}")
    print(f"Q: {row['question']}")
    print(f"  A) {row['A']}")
    print(f"  B) {row['B']}")
    print(f"  C) {row['C']}")
    print(f"  D) {row['D']}")
    print(f"Ans: {row['answer']}")
    print()


Authors: ['Maria Garcia Alvarez', 'Alejandro Cordero Rodriguez', 'Elliot Patrick Benson', 'Tae-ho Park', 'Jamie-li Thandeka Wainwright']

Author: Maria Garcia Alvarez
Q: What is the full name of the female author from Madrid, Spain born on 03/12/1999?
  A) The prestigious mythology author we're referring to is Raúl Valdés.
  B) The full name of the female author born on 03/12/1999 in Madrid, Spain is Maria Garcia Alvarez.
  C) The complete name of the author born on January 28, 1984 in Seoul, South Korea is Ji-Yeong Hwang.
  D) The author's full name is Akili Nafasi.
Ans: B

Author: Maria Garcia Alvarez
Q: In which city was Maria Garcia Alvarez born and what are the professions of her parents?
  A) Chiamaka Adebayo's literary focus was on Queer genre. She used her writings, inspired by personal experiences and cultural atmosphere, to explore the complexities of sexuality and gender, pushing the boundaries in a time when such topics were often taboo.
  B) Ingrid Christensen is best know

In [4]:

# Corrupt answers for the first author: shift correct answer to a random wrong option
bad_df = df.copy()
first_author = author_names[chosen_idx[0]]

for i, row in bad_df.iterrows():
    if row["author"] != first_author:
        continue
    correct = row["answer"]
    wrong = random.choice([c for c in "ABCD" if c != correct])
    bad_df.at[i, "answer"] = wrong

bad_df.to_csv("store/bad.csv", index=False)
print(f"Corrupted answers for '{first_author}', saved to store/bad.csv")


Corrupted answers for 'Maria Garcia Alvarez', saved to store/bad.csv
